<a href="https://colab.research.google.com/github/chaunijs/onlineshoppingprice/blob/main/7_eleven_allonline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# @title install command
import subprocess
import sys
from IPython.display import display, HTML

# 1. Setup the 2-line scrolling box UI
display(HTML("""
    <style>
        #scroll_output {
            height: 50px;
            overflow-y: scroll;
            background-color: #1e1e1e;
            color: #00ff00;
            padding: 10px;
            font-family: monospace;
            font-size: 14px;
            border: 1px solid #444;
            display: flex;
            flex-direction: column;
        }
    </style>
    <div id="scroll_output">Starting Makro environment setup...</div>
"""))

def run_and_scroll(commands):
    """Runs list of commands and streams output to the scroll_output div"""
    for cmd in commands:
        process = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            escaped_line = line.replace("'", "\\'").strip()
            display(HTML(f"""<script>
                var obj = document.getElementById('scroll_output');
                obj.innerHTML += '<div>' + '{escaped_line}' + '</div>';
                obj.scrollTop = obj.scrollHeight;
            </script>"""), display_id='scroll_update')
        process.wait()

# 2. Consolidated commands for Makro, Scrapling, OCR, and Manual Chrome Installation
commands_to_run = [
    "apt-get update",
    # Download the latest Chrome Stable directly from Google
    "wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb",
    # Install the downloaded deb file alongside Chromium dependencies
    "apt-get install -y ./google-chrome-stable_current_amd64.deb chromium-browser chromium-chromedriver",
    # One big PIP install for all libraries (Scraping + Polars + OCR + Excel + Webdriver Management)
    "pip install selenium bs4 beautifulsoup4 pandas polars xlsxwriter fastexcel curl_cffi scrapling playwright patchright msgspec browserforge nest_asyncio itables chromedriver-autoinstaller webdriver-manager google-colab-selenium easyocr pillow requests -q",
    # Setup the stealth browser
    "patchright install chromium",
    "patchright install-deps"
]

run_and_scroll(commands_to_run)
print("\n✅ Environment ready! Let's go.")


✅ Environment ready! Let's go.


In [2]:
# from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
import polars as pl
import asyncio
import datetime
from datetime import date
import IPython.display as display
import datetime
import xlsxwriter
from google.colab import data_table
data_table.enable_dataframe_formatter()
today_date = datetime.datetime.now().strftime("%Y-%m-%d")
print("Today is",today_date)

Today is 2026-05-06


In [3]:
# @title scrape_7eleven_data (Final Container Logic)
import time
import re
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import polars as pl

def scrape_7eleven_data(base_url: str) -> pl.DataFrame:
    """
    Scrapes product data from 7-Eleven's online store using strict container boundaries.
    """
    chrome_options = Options()
    chrome_options.add_argument("--headless=new")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")

    print("Starting browser...")
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=chrome_options)
    driver.set_window_size(1920, 1080)

    data = []
    p_index = 0
    previous_page_names = []

    while True:
        print(f"\nScraping Page {p_index + 1} (URL parameter p={p_index})...")

        current_page_url = base_url + f"&p={p_index}"
        driver.get(current_page_url)

        try:
            # Wait for the main item wrapper from your screenshot to load
            WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.CLASS_NAME, "item-list-wrapper"))
            )
        except Exception as e:
            print("No products found on this page. Reached the end!")
            break

        soup = BeautifulSoup(driver.page_source, "html.parser")

        # 1. Grab ALL product containers (Based on your screenshot!)
        product_cards = soup.select(".item-list-wrapper, .product-item")

        if not product_cards:
            print("Page is empty. Reached the end!")
            break

        # Check for duplicates to prevent infinite loops
        current_page_names = []
        for card in product_cards:
            # The title is often stored securely in the <a> tag attribute 'title'
            a_tag = card.select_one("a.productlink")
            name = a_tag.get("title") if a_tag and a_tag.has_attr("title") else None

            # Fallback if title attribute isn't there
            if not name:
                desc_elem = card.select_one(".item-description-cls-mobile")
                name = desc_elem.text.strip() if desc_elem else "Unknown"

            current_page_names.append(name)

        if current_page_names == previous_page_names:
            print("Duplicate items detected! We have reached the final page.")
            break

        previous_page_names = current_page_names.copy()

        # 2. Parse Data strictly within each card
        for card, name in zip(product_cards, current_page_names):

            # Prices
            promo_elem = card.select_one("strong")
            orig_elem = card.select_one("s, strike, del")

            promotion_price = promo_elem.text.strip() if promo_elem else None
            original_price = orig_elem.text.strip() if orig_elem else None

            # 3. Flag Extraction (Scoped to this specific item box)
            condition = None
            flag_elem = card.select_one(".flag")

            if flag_elem and flag_elem.has_attr("style"):
                match = re.search(r'\/([A-Za-z0-9_]+)-th\.svg', flag_elem["style"])
                if match:
                    condition = match.group(1).upper()

            data.append({
                "name": name,
                "promotion_price": promotion_price,
                "original_price": original_price,
                "condition": condition
            })

        print(f"Successfully grabbed {len(product_cards)} items from Page {p_index + 1}.")
        p_index += 1

    driver.quit()

    # Clean Data using Polars
    df = pl.DataFrame(
        data,
        schema={
            "name": str,
            "promotion_price": str,
            "original_price": str,
            "condition": str
        }
    )

    if not df.is_empty():
        df = df.with_columns(
            pl.col("promotion_price").str.replace_all(r"[^\d.]", "").cast(pl.Float64, strict=False).alias("promotion_price"),
            pl.col("original_price").str.replace_all(r"[^\d.]", "").cast(pl.Float64, strict=False).alias("original_price")
        )

        df = df.unique(subset=['name'], keep='first')

        print("\n--- Final Scraping Results ---")
        print(df.tail(10))
        print(f"\nFinal clean item count: {df.height}")

        return df
    else:
        print("No data was scraped.")
        return pl.DataFrame()

# --- Execution ---
list_of_7eleven_urls = [
    "https://allonline.7eleven.co.th/supermarket/household-items/laundry/liquid-detergent/?filter.BRAND=Fineline&filter.BRAND=Hygiene&filter.BRAND=%E0%B9%80%E0%B8%9B%E0%B8%B2&filter.BRAND=%E0%B9%81%E0%B8%AD%E0%B8%97%E0%B9%81%E0%B8%97%E0%B8%84&filter.PRICE=49-470&filter.initial_from_PRICE=49&filter.initial_to_PRICE=470&landing=true&pageSize=90&sortBy=si&view=0",

    "https://allonline.7eleven.co.th/supermarket/household-items/dish-detergent/?show=all&filter.from_PRICE=53&filter.to_PRICE=899&brands=on&sortBy=si&filter.BRAND=%E0%B9%84%E0%B8%A5%E0%B8%9B%E0%B8%AD%E0%B8%99%E0%B9%80%E0%B8%AD%E0%B8%9F&filter.initial_from_PRICE=53&filter.initial_to_PRICE=899&view=6",
]

all_scraped_dataframes = []

for url in list_of_7eleven_urls:
    print(f"\n--- Starting scraping for URL: {url} ---")
    df_current_url = scrape_7eleven_data(url)
    if not df_current_url.is_empty():
        all_scraped_dataframes.append(df_current_url)

if all_scraped_dataframes:
    df_combined_7eleven = pl.concat(all_scraped_dataframes, how='vertical')
    df_combined_7eleven = df_combined_7eleven.unique(subset=['name'], keep='first')
    print(f"\nCombined data after dropping duplicates. Final unique item count: {df_combined_7eleven.height}")
else:
    print("No data was scraped from any of the provided URLs.")


--- Starting scraping for URL: https://allonline.7eleven.co.th/supermarket/household-items/laundry/liquid-detergent/?filter.BRAND=Fineline&filter.BRAND=Hygiene&filter.BRAND=%E0%B9%80%E0%B8%9B%E0%B8%B2&filter.BRAND=%E0%B9%81%E0%B8%AD%E0%B8%97%E0%B9%81%E0%B8%97%E0%B8%84&filter.PRICE=49-470&filter.initial_from_PRICE=49&filter.initial_to_PRICE=470&landing=true&pageSize=90&sortBy=si&view=0 ---
Starting browser...

Scraping Page 1 (URL parameter p=0)...
Successfully grabbed 180 items from Page 1.

Scraping Page 2 (URL parameter p=1)...
Successfully grabbed 12 items from Page 2.

Scraping Page 3 (URL parameter p=2)...
Duplicate items detected! We have reached the final page.

--- Final Scraping Results ---
shape: (10, 4)
┌─────────────────────────────┬─────────────────┬────────────────┬───────────┐
│ name                        ┆ promotion_price ┆ original_price ┆ condition │
│ ---                         ┆ ---             ┆ ---            ┆ ---       │
│ str                         ┆ f64   

In [4]:
df_combined_7eleven.to_pandas()

,name,promotion_price,original_price,condition
0,"เปา เจลแคป 18 ชิ้น - เปา, น้ำยาซักผ้า",199.0,NaN,None
1,ไฟน์ไลน์ น้ำยาซักผ้า สูตรเข้มข้น ดีลักซ์เพอร์ฟ...,69.0,89.0,None
2,ไฮยีน คัลเลอร์ ผลิตภัณฑ์ซักผ้าขาว 500 มล. - Hy...,110.0,117.0,None
3,ไฮยีน เอ็กซ์เพิร์ท วอช ผลิตภัณฑ์ซักผ้าชนิดน้ำ ...,139.0,NaN,None
4,เปาวินวอชน้ำยาซักผ้าลิควิด 600 มล. (แพ็กคู่) -...,139.0,NaN,None
...,...,...,...,...
98,เปา วินวอชลิควิดไวท์ฟลอรั่ลถุงเติม น้ำยาซักผ้า...,99.0,NaN,None
99,แอทแทค น้ำยาซักผ้ากลิ่นชาร์มมิ่ง โรมานซ์ 1300 ...,159.0,180.0,None
100,ไฮยีน น้ำยาซักผ้า แฮปปี้ซันชายน์ 600 มล. - Hyg...,65.0,NaN,None
101,เปาวินวอช น้ำยาซักผ้า ลิควิด สีฟ้า 620 มล. - เ...,189.0,198.0,None


In [18]:
# @title Scrape Specific WatchList Product URL (Targeted HTML Parser)
import time
import re
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import polars as pl

def get_driver():
    """Helper to initialize the Selenium driver."""
    chrome_options = Options()
    chrome_options.add_argument("--headless=new")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=chrome_options)
    driver.set_window_size(1920, 1080)
    return driver

def scrape_specific_products(urls: list) -> pl.DataFrame:
    """
    Scrapes data from 7-Eleven detail pages using strictly targeted CSS selectors.
    """
    driver = get_driver()
    data = []

    for url in urls:
        print(f"Scraping product: {url}")
        try:
            driver.get(url)

            # Wait specifically for the .currentPrice class to render
            try:
                WebDriverWait(driver, 15).until(
                    EC.presence_of_element_located((By.CSS_SELECTOR, ".currentPrice, h1"))
                )
            except Exception:
                time.sleep(3)

            soup = BeautifulSoup(driver.page_source, "html.parser")

            # 1. Extract Product Name
            name_elem = soup.find("h1")
            name = name_elem.text.strip() if name_elem else "Unknown Name"

            # 2. Extract Prices
            promo_elem = soup.select_one(".currentPrice")
            original_elem = soup.select_one("strike")

            promo_text = promo_elem.text.strip() if promo_elem else None
            orig_text = original_elem.text.strip() if original_elem else None

            # 3. Extract Condition Flag (Scoped exactly to your screenshot)
            condition = None

            # The screenshot shows the main image flag is inside a "gallery" or "art-detail-page-top" block.
            # We isolate this section so we don't accidentally scan the footer products.
            gallery_area = soup.select_one(".gallery, .art-detail-page-top")
            search_area = gallery_area if gallery_area else soup

            # Find all elements with the 'flag' class inside this specific area
            flag_elems = search_area.select(".flag")

            for flag in flag_elems:
                style_attr = flag.get("style", "").upper()
                if "1GET1" in style_attr:
                    condition = "1GET1"
                    break
                elif "2GET1" in style_attr:
                    condition = "2GET1"
                    break

            data.append({
                "name": name,
                "promotion_price": promo_text,
                "original_price": orig_text,
                "url": url,
                "condition": condition
            })

            print(f"  -> Promo: {promo_text} | Orig: {orig_text} | Flag: {condition}")

        except Exception as e:
            print(f"  Error scraping {url}: {str(e)[:100]}...")

        time.sleep(2)

    driver.quit()

    # Clean Data using Polars
    df = pl.DataFrame(
        data,
        schema={
            "name": str,
            "promotion_price": str,
            "original_price": str,
            "url": str,
            "condition": str
        }
    )

    if not df.is_empty():
        df = df.with_columns(
            pl.col("promotion_price").str.replace_all(r"[^\d.]", "").cast(pl.Float64, strict=False),
            pl.col("original_price").str.replace_all(r"[^\d.]", "").cast(pl.Float64, strict=False)
        )

    return df

# --- Execution ---
specific_product_urls = [
    # ไลปอนเอฟ น้ำยาล้างจาน สูตรอนามัย 3200 มล.
    "https://www.allonline.7eleven.co.th/p/%E0%B9%84%E0%B8%A5%E0%B8%9B%E0%B8%AD%E0%B8%99%E0%B9%80%E0%B8%AD%E0%B8%9F-%E0%B8%99%E0%B9%89%E0%B8%B3%E0%B8%A2%E0%B8%B2%E0%B8%A5%E0%B9%89%E0%B8%B2%E0%B8%87%E0%B8%88%E0%B8%B2%E0%B8%99-%E0%B8%AA%E0%B8%B9%E0%B8%95%E0%B8%A3%E0%B8%AD%E0%B8%99%E0%B8%B2%E0%B8%A1%E0%B8%B1%E0%B8%A2-3200-%E0%B8%A1%E0%B8%A5/473248/",
    # ไลปอนเอฟ น้ำยาล้างจาน สูตรอนามัย 500 มล. (แพ็ก 3 ชิ้น)
    "https://www.allonline.7eleven.co.th/p/%E0%B9%84%E0%B8%A5%E0%B8%9B%E0%B8%AD%E0%B8%99%E0%B9%80%E0%B8%AD%E0%B8%9F-%E0%B8%99%E0%B9%89%E0%B8%B3%E0%B8%A2%E0%B8%B2%E0%B8%A5%E0%B9%89%E0%B8%B2%E0%B8%87%E0%B8%88%E0%B8%B2%E0%B8%99-%E0%B8%AA%E0%B8%B9%E0%B8%95%E0%B8%A3%E0%B8%AD%E0%B8%99%E0%B8%B2%E0%B8%A1%E0%B8%B1%E0%B8%A2-500-%E0%B8%A1%E0%B8%A5-%E0%B9%81%E0%B8%9E%E0%B9%87%E0%B8%81-3-%E0%B8%8A%E0%B8%B4%E0%B9%89%E0%B8%99/456259/?p=0&q=%E0%B9%84%E0%B8%A5%E0%B8%9B%E0%B8%AD%E0%B8%99%E0%B9%80%E0%B8%AD%E0%B8%9F%20%E0%B8%99%E0%B9%89%E0%B8%B3%E0%B8%A2%E0%B8%B2%E0%B8%A5%E0%B9%89%E0%B8%B2%E0%B8%87%E0%B8%88%E0%B8%B2%E0%B8%99%20%E0%B8%AA%E0%B8%B9%E0%B8%95%E0%B8%A3%E0%B8%AD%E0%B8%99%E0%B8%B2%E0%B8%A1%E0%B8%B1%E0%B8%A2%20500%20%E0%B8%A1%E0%B8%A5.%20%28%E0%B9%81%E0%B8%9E%E0%B9%87%E0%B8%81%203%20%E0%B8%8A%E0%B8%B4%E0%B9%89%E0%B8%99%29&view=0&requestId=f0oJr7cdQlWZddudy6-uAg&categoryId=178730951#itemId=356664_0",
    # ไฟน์ไลน์ พลัส น้ำยาซักผ้า ซันนี่ โกลด์ 1250 มล.
    "https://allonline.7eleven.co.th/p/%E0%B9%84%E0%B8%9F%E0%B8%99%E0%B9%8C%E0%B9%84%E0%B8%A5%E0%B8%99%E0%B9%8C-%E0%B8%9E%E0%B8%A5%E0%B8%B1%E0%B8%AA-%E0%B8%99%E0%B9%89%E0%B8%B3%E0%B8%A2%E0%B8%B2%E0%B8%8B%E0%B8%B1%E0%B8%81%E0%B8%9C%E0%B9%89%E0%B8%B2-%E0%B8%8B%E0%B8%B1%E0%B8%99%E0%B8%99%E0%B8%B5%E0%B9%88-%E0%B9%82%E0%B8%81%E0%B8%A5%E0%B8%94%E0%B9%8C-1250-%E0%B8%A1%E0%B8%A5/346148/",
    # ไฟน์ไลน์พลัสซักผ้าชนิดน้ำสีทอง 550 มล.
    "https://www.allonline.7eleven.co.th/p/%E0%B9%84%E0%B8%9F%E0%B8%99%E0%B9%8C%E0%B9%84%E0%B8%A5%E0%B8%99%E0%B9%8C%E0%B8%9E%E0%B8%A5%E0%B8%B1%E0%B8%AA%E0%B8%8B%E0%B8%B1%E0%B8%81%E0%B8%9C%E0%B9%89%E0%B8%B2%E0%B8%8A%E0%B8%99%E0%B8%B4%E0%B8%94%E0%B8%99%E0%B9%89%E0%B8%B3%E0%B8%AA%E0%B8%B5%E0%B8%97%E0%B8%AD%E0%B8%87-550-%E0%B8%A1%E0%B8%A5/462765",
    # ไฮยีน น้ำยาซักผ้า มิลค์กี้ ทัช 600 มล.
    "https://www.allonline.7eleven.co.th/p/%E0%B9%84%E0%B8%AE%E0%B8%A2%E0%B8%B5%E0%B8%99-%E0%B8%99%E0%B9%89%E0%B8%B3%E0%B8%A2%E0%B8%B2%E0%B8%8B%E0%B8%B1%E0%B8%81%E0%B8%9C%E0%B9%89%E0%B8%B2-%E0%B8%A1%E0%B8%B4%E0%B8%A5%E0%B8%84%E0%B9%8C%E0%B8%81%E0%B8%B5%E0%B9%89-%E0%B8%97%E0%B8%B1%E0%B8%8A-600-%E0%B8%A1%E0%B8%A5/351927",
    # ไฮยีน เอ็กซ์เพิร์ท วอช น้ำยาซักผ้า มิลค์กี้ ทัช 1400 มล.
    "https://www.allonline.7eleven.co.th/p/%E0%B9%84%E0%B8%AE%E0%B8%A2%E0%B8%B5%E0%B8%99-%E0%B9%80%E0%B8%AD%E0%B9%87%E0%B8%81%E0%B8%8B%E0%B9%8C%E0%B9%80%E0%B8%9E%E0%B8%B4%E0%B8%A3%E0%B9%8C%E0%B8%97-%E0%B8%A7%E0%B8%AD%E0%B8%8A-%E0%B8%99%E0%B9%89%E0%B8%B3%E0%B8%A2%E0%B8%B2%E0%B8%8B%E0%B8%B1%E0%B8%81%E0%B8%9C%E0%B9%89%E0%B8%B2-%E0%B8%A1%E0%B8%B4%E0%B8%A5%E0%B8%84%E0%B9%8C%E0%B8%81%E0%B8%B5%E0%B9%89-%E0%B8%97%E0%B8%B1%E0%B8%8A-1400-%E0%B8%A1%E0%B8%A5/376426",
    # ไฮยีน น้ำยาปรับผ้านุ่ม มิลค์กี้ทัช 480 มล.
    "https://allonline.7eleven.co.th/p/%E0%B9%84%E0%B8%AE%E0%B8%A2%E0%B8%B5%E0%B8%99-%E0%B8%99%E0%B9%89%E0%B8%B3%E0%B8%A2%E0%B8%B2%E0%B8%9B%E0%B8%A3%E0%B8%B1%E0%B8%9A%E0%B8%9C%E0%B9%89%E0%B8%B2%E0%B8%99%E0%B8%B8%E0%B9%88%E0%B8%A1-%E0%B8%A1%E0%B8%B4%E0%B8%A5%E0%B8%84%E0%B9%8C%E0%B8%81%E0%B8%B5%E0%B9%89%E0%B8%97%E0%B8%B1%E0%B8%8A-480-%E0%B8%A1%E0%B8%A5/328226/",

    # ไฮยีน เอ็กซ์เพิร์ทแคร์ น้ำยาปรับผ้านุ่ม ขาวมิลค์กี้ 1000 มล.
    "https://allonline.7eleven.co.th/p/%E0%B9%84%E0%B8%AE%E0%B8%A2%E0%B8%B5%E0%B8%99-%E0%B9%80%E0%B8%AD%E0%B9%87%E0%B8%81%E0%B8%8B%E0%B9%8C%E0%B9%80%E0%B8%9E%E0%B8%B4%E0%B8%A3%E0%B9%8C%E0%B8%97%E0%B9%81%E0%B8%84%E0%B8%A3%E0%B9%8C-%E0%B8%99%E0%B9%89%E0%B8%B3%E0%B8%A2%E0%B8%B2%E0%B8%9B%E0%B8%A3%E0%B8%B1%E0%B8%9A%E0%B8%9C%E0%B9%89%E0%B8%B2%E0%B8%99%E0%B8%B8%E0%B9%88%E0%B8%A1-%E0%B8%82%E0%B8%B2%E0%B8%A7%E0%B8%A1%E0%B8%B4%E0%B8%A5%E0%B8%84%E0%B9%8C%E0%B8%81%E0%B8%B5%E0%B9%89-1000-%E0%B8%A1%E0%B8%A5/467880",

    # เปาวินวอช น้ำยาซักผ้าลิควิด 620 มล.
    "https://www.allonline.7eleven.co.th/p/%E0%B9%80%E0%B8%9B%E0%B8%B2%E0%B8%A7%E0%B8%B4%E0%B8%99%E0%B8%A7%E0%B8%AD%E0%B8%8A-%E0%B8%99%E0%B9%89%E0%B8%B3%E0%B8%A2%E0%B8%B2%E0%B8%8B%E0%B8%B1%E0%B8%81%E0%B8%9C%E0%B9%89%E0%B8%B2%E0%B8%A5%E0%B8%B4%E0%B8%84%E0%B8%A7%E0%B8%B4%E0%B8%94-620-%E0%B8%A1%E0%B8%A5/450704",
    # เปา ไวท์นาโนเทค ผงซักฟอก 2400 กรัม
    "https://allonline.7eleven.co.th/p/%E0%B9%80%E0%B8%9B%E0%B8%B2-%E0%B9%84%E0%B8%A7%E0%B8%97%E0%B9%8C%E0%B8%99%E0%B8%B2%E0%B9%82%E0%B8%99%E0%B9%80%E0%B8%97%E0%B8%84%E0%B8%9C%E0%B8%87%E0%B8%8B%E0%B8%B1%E0%B8%81%E0%B8%9F%E0%B8%AD%E0%B8%812400-%E0%B8%81%E0%B8%A3%E0%B8%B1%E0%B8%A1/462743",
    # แอทแทค ผงซักฟอก แฮปปี้ สวีท 1800 กรัม (แพ็กคู่)
    "https://allonline.7eleven.co.th/p/%E0%B9%81%E0%B8%AD%E0%B8%97%E0%B9%81%E0%B8%97%E0%B8%84-%E0%B8%9C%E0%B8%87%E0%B8%8B%E0%B8%B1%E0%B8%81%E0%B8%9F%E0%B8%AD%E0%B8%81-%E0%B9%81%E0%B8%AE%E0%B8%9B%E0%B8%9B%E0%B8%B5%E0%B9%89-%E0%B8%AA%E0%B8%A7%E0%B8%B5%E0%B8%97-1800-%E0%B8%81%E0%B8%A3%E0%B8%B1%E0%B8%A1-%E0%B9%81%E0%B8%9E%E0%B9%87%E0%B8%81%E0%B8%84%E0%B8%B9%E0%B9%88/346501",
    # แอทแทค ผงซักฟอก แฮปปี้สวีท 2500 กรัม
    "https://allonline.7eleven.co.th/p/%E0%B9%81%E0%B8%AD%E0%B8%97%E0%B9%81%E0%B8%97%E0%B8%84-%E0%B8%9C%E0%B8%87%E0%B8%8B%E0%B8%B1%E0%B8%81%E0%B8%9F%E0%B8%AD%E0%B8%81-%E0%B9%81%E0%B8%AE%E0%B8%9B%E0%B8%9B%E0%B8%B5%E0%B9%89%E0%B8%AA%E0%B8%A7%E0%B8%B5%E0%B8%97-2500-%E0%B8%81%E0%B8%A3%E0%B8%B1%E0%B8%A1/334941",
    # โปร บลูพลัส ผงซักฟอก 2400 กรัม
    "https://allonline.7eleven.co.th/p/%E0%B9%82%E0%B8%9B%E0%B8%A3-%E0%B8%9A%E0%B8%A5%E0%B8%B9%E0%B8%9E%E0%B8%A5%E0%B8%B1%E0%B8%AA-%E0%B8%9C%E0%B8%87%E0%B8%8B%E0%B8%B1%E0%B8%81%E0%B8%9F%E0%B8%AD%E0%B8%81-2400-%E0%B8%81%E0%B8%A3%E0%B8%B1%E0%B8%A1/462744",

    # เปา วินวอชลิควิด ถุงเติม น้ำยาซักผ้า 1500 มล.
    "https://www.allonline.7eleven.co.th/p/%E0%B9%80%E0%B8%9B%E0%B8%B2-%E0%B8%A7%E0%B8%B4%E0%B8%99%E0%B8%A7%E0%B8%AD%E0%B8%8A%E0%B8%A5%E0%B8%B4%E0%B8%84%E0%B8%A7%E0%B8%B4%E0%B8%94-%E0%B8%96%E0%B8%B8%E0%B8%87%E0%B9%80%E0%B8%95%E0%B8%B4%E0%B8%A1%E0%B8%99%E0%B9%89%E0%B8%B3%E0%B8%A2%E0%B8%B2%E0%B8%8B%E0%B8%B1%E0%B8%81%E0%B8%9C%E0%B9%89%E0%B8%B21500-%E0%B8%A1%E0%B8%A5/509381/",


    # -------------Control 1 GET 1-------------------
    # ไฟน์ไลน์ ดีลักซ์ เพอร์ฟูม ปรับผ้านุ่มสูตรเข้มข้นพิเศษ เทนเดอร์ เซนท์ ทอง 490 มล.
    "https://www.allonline.7eleven.co.th/p/%E0%B9%84%E0%B8%9F%E0%B8%99%E0%B9%8C%E0%B9%84%E0%B8%A5%E0%B8%99%E0%B9%8C-%E0%B8%94%E0%B8%B5%E0%B8%A5%E0%B8%B1%E0%B8%81%E0%B8%8B%E0%B9%8C-%E0%B9%80%E0%B8%9E%E0%B8%AD%E0%B8%A3%E0%B9%8C%E0%B8%9F%E0%B8%B9%E0%B8%A1-%E0%B8%9B%E0%B8%A3%E0%B8%B1%E0%B8%9A%E0%B8%9C%E0%B9%89%E0%B8%B2%E0%B8%99%E0%B8%B8%E0%B9%88%E0%B8%A1%E0%B8%AA%E0%B8%B9%E0%B8%95%E0%B8%A3%E0%B9%80%E0%B8%82%E0%B9%89%E0%B8%A1%E0%B8%82%E0%B9%89%E0%B8%99%E0%B8%9E%E0%B8%B4%E0%B9%80%E0%B8%A8%E0%B8%A9-%E0%B9%80%E0%B8%97%E0%B8%99%E0%B9%80%E0%B8%94%E0%B8%AD%E0%B8%A3%E0%B9%8C-%E0%B9%80%E0%B8%8B%E0%B8%99%E0%B8%97%E0%B9%8C-%E0%B8%97%E0%B8%AD%E0%B8%87-490-%E0%B8%A1%E0%B8%A5/343673/",
]

df_watchlist = scrape_specific_products(specific_product_urls)
print("\n--- Final Results ---")

Scraping product: https://www.allonline.7eleven.co.th/p/%E0%B9%84%E0%B8%A5%E0%B8%9B%E0%B8%AD%E0%B8%99%E0%B9%80%E0%B8%AD%E0%B8%9F-%E0%B8%99%E0%B9%89%E0%B8%B3%E0%B8%A2%E0%B8%B2%E0%B8%A5%E0%B9%89%E0%B8%B2%E0%B8%87%E0%B8%88%E0%B8%B2%E0%B8%99-%E0%B8%AA%E0%B8%B9%E0%B8%95%E0%B8%A3%E0%B8%AD%E0%B8%99%E0%B8%B2%E0%B8%A1%E0%B8%B1%E0%B8%A2-3200-%E0%B8%A1%E0%B8%A5/473248/
  -> Promo: ฿ 165 | Orig: None | Flag: None
Scraping product: https://www.allonline.7eleven.co.th/p/%E0%B9%84%E0%B8%A5%E0%B8%9B%E0%B8%AD%E0%B8%99%E0%B9%80%E0%B8%AD%E0%B8%9F-%E0%B8%99%E0%B9%89%E0%B8%B3%E0%B8%A2%E0%B8%B2%E0%B8%A5%E0%B9%89%E0%B8%B2%E0%B8%87%E0%B8%88%E0%B8%B2%E0%B8%99-%E0%B8%AA%E0%B8%B9%E0%B8%95%E0%B8%A3%E0%B8%AD%E0%B8%99%E0%B8%B2%E0%B8%A1%E0%B8%B1%E0%B8%A2-500-%E0%B8%A1%E0%B8%A5-%E0%B9%81%E0%B8%9E%E0%B9%87%E0%B8%81-3-%E0%B8%8A%E0%B8%B4%E0%B9%89%E0%B8%99/456259/?p=0&q=%E0%B9%84%E0%B8%A5%E0%B8%9B%E0%B8%AD%E0%B8%99%E0%B9%80%E0%B8%AD%E0%B8%9F%20%E0%B8%99%E0%B9%89%E0%B8%B3%E0%B8%A2%E0%B8%B2%E0%B8%A5%E0%B9%89%E0%B8%B2%E0%B8

In [25]:
df_watchlist.to_pandas()

,name,promotion_price,original_price,url,condition
0,ไลปอนเอฟ น้ำยาล้างจาน สูตรอนามัย 3200 มล.,165.0,NaN,https://www.allonline.7eleven.co.th/p/%E0%B9%8...,None
1,Unknown Name,NaN,NaN,https://www.allonline.7eleven.co.th/p/%E0%B9%8...,None
2,ไฟน์ไลน์ พลัส น้ำยาซักผ้า ซันนี่ โกลด์ 1250 มล.,149.0,179.0,https://allonline.7eleven.co.th/p/%E0%B9%84%E0...,None
3,ไฟน์ไลน์พลัสซักผ้าชนิดน้ำสีทอง 550 มล.,69.0,89.0,https://www.allonline.7eleven.co.th/p/%E0%B9%8...,None
4,ไฮยีน น้ำยาซักผ้า มิลค์กี้ ทัช 600 มล.,65.0,NaN,https://www.allonline.7eleven.co.th/p/%E0%B9%8...,None
5,ไฮยีน เอ็กซ์เพิร์ท วอช น้ำยาซักผ้า มิลค์กี้ ทั...,139.0,NaN,https://www.allonline.7eleven.co.th/p/%E0%B9%8...,None
6,ไฮยีน น้ำยาปรับผ้านุ่ม มิลค์กี้ทัช 480 มล.,114.0,120.0,https://allonline.7eleven.co.th/p/%E0%B9%84%E0...,None
7,ไฮยีน เอ็กซ์เพิร์ทแคร์ น้ำยาปรับผ้านุ่ม ขาวมิล...,129.0,NaN,https://allonline.7eleven.co.th/p/%E0%B9%84%E0...,None
8,เปาวินวอช น้ำยาซักผ้าลิควิด 620 มล.,189.0,198.0,https://www.allonline.7eleven.co.th/p/%E0%B9%8...,None
9,เปา ไวท์นาโนเทค ผงซักฟอก 2400 กรัม,105.0,155.0,https://allonline.7eleven.co.th/p/%E0%B9%80%E0...,None


In [28]:
cols_sel_before_process = ["name","promotion_price","original_price", "condition"]
df_watchlist_sel= df_watchlist.select(cols_sel_before_process)
just_watchlist = True
if just_watchlist:
  df_combined_7eleven_sel = df_watchlist_sel
else:
  df_combined_7eleven_sel =df_combined_7eleven.select(cols_sel_before_process)
  df_combined_7eleven_sel = pl.concat([df_combined_7eleven_sel, df_watchlist_sel])

In [8]:
df_combined_7eleven_sel

name,promotion_price,original_price,condition
str,f64,f64,str
"""ไลปอนเอฟ น้ำยาล้างจาน สูตรอนาม…",165.0,null,null
"""Unknown Name""",null,null,null
"""ไฟน์ไลน์ พลัส น้ำยาซักผ้า ซันน…",149.0,179.0,null
"""ไฟน์ไลน์พลัสซักผ้าชนิดน้ำสีทอง…",69.0,89.0,null
"""ไฮยีน น้ำยาซักผ้า มิลค์กี้ ทัช…",65.0,null,null
…,…,…,…
"""เปา ไวท์นาโนเทค ผงซักฟอก 2400 …",105.0,155.0,null
"""Unknown Name""",null,null,null
"""แอทแทค ผงซักฟอก แฮปปี้สวีท 250…",310.0,null,null


In [9]:
# @title udf data-prep
import polars as pl

def re_evaluate_price(df: pl.DataFrame) -> pl.DataFrame:
    """
    Standardizes pricing logic:
    1. If original_price is missing, move the promotion_price to it.
    2. If promotion_price matches the original_price, set it to Null.
    """
    return (
        df.with_columns(
            # Step 1: Fix missing original_prices by 'swapping' from promotion_price
            pl.when(pl.col("original_price").is_null() & pl.col("promotion_price").is_not_null())
            .then(pl.col("promotion_price"))
            .otherwise(pl.col("original_price"))
            .alias("original_price")
        )
        .with_columns(
            # Step 2: Now that original_price is populated, nullify redundant promotions
            pl.when(pl.col("promotion_price") == pl.col("original_price"))
            .then(None)
            .otherwise(pl.col("promotion_price"))
            .alias("promotion_price")
        )
    )

# Apply the preparation function
df_prep_7_eleven = re_evaluate_price(df_combined_7eleven_sel)

print("Data has been re-evaluated.")


Data has been re-evaluated.


In [10]:
df_prep_7_eleven

name,promotion_price,original_price,condition
str,f64,f64,str
"""ไลปอนเอฟ น้ำยาล้างจาน สูตรอนาม…",null,165.0,null
"""Unknown Name""",null,null,null
"""ไฟน์ไลน์ พลัส น้ำยาซักผ้า ซันน…",149.0,179.0,null
"""ไฟน์ไลน์พลัสซักผ้าชนิดน้ำสีทอง…",69.0,89.0,null
"""ไฮยีน น้ำยาซักผ้า มิลค์กี้ ทัช…",null,65.0,null
…,…,…,…
"""เปา ไวท์นาโนเทค ผงซักฟอก 2400 …",105.0,155.0,null
"""Unknown Name""",null,null,null
"""แอทแทค ผงซักฟอก แฮปปี้สวีท 250…",null,310.0,null


In [11]:
import polars as pl
from datetime import date
import re

def parse_product_names_TH(df: pl.DataFrame, shop_name: str) -> pl.DataFrame:
    """
    Standardizes supermarket dataframes.
    Updated to include 'กรัม' and handle Thai unit formatting.
    """
    # 1. Setup the dynamic date
    today_date = date.today().strftime("%Y-%m-%d")

    # 2. Updated patterns
    # Added 'กรัม' to the unit group.
    # Note: We use \s* to handle potential spaces between number and unit.
    quant_unit_pattern = r"(?i)([\d,.]+)\s*(มล\.|ลิตร|ก\.ก\.|กรัม|ML|G|KG|L)"

    pack_pattern = r"(?i)(PACK\s*\d*\s*FREE\s*\d+|PACK\s*\d*\s*\+\s*\d+|PACK\s*\d+|TWINPACK|\bX\s*\d+\b|P?\d+\s*\+\s*\d+|\(\d+\+\d+\)|\d+\s*FREE\s*\d+|\bPACK\b|แพ็ก\s*\d+\s*ชิ้น)"

    # Thai brand extraction logic
    thai_brands = ["ไฟน์ไลน์", "ไฮยีน", "เปา", "แอทแทค", "ไลปอนเอฟ"]
    brand_pattern = r"^(" + "|".join(re.escape(b) for b in thai_brands) + r")"

    # 3. Apply the Polars transformation
    return df.with_columns([
        pl.lit(None).alias("condition"),
        pl.lit(today_date).alias("Date"),

        # Extract Brand
        pl.col("name").str.extract(brand_pattern, 1)
          .fill_null(pl.col("name").str.split(" ").list.first())
          .alias("Brand"),

        # Fixed Volume Extraction (handles decimals/commas before casting)
        pl.col("name")
          .str.extract(quant_unit_pattern, 1)
          .str.replace_all(",", "")
          .cast(pl.Float64, strict=False) # Changed to Float64 in case of "1.5" Liters
          .alias("Volume"),

        # Extract Unit
        pl.col("name")
          .str.extract(quant_unit_pattern, 2)
          .alias("Unit"),

        # Extract Pack size
        pl.col("name")
          .str.extract(pack_pattern, 0) # Use group 0 to get the whole match
          .str.to_uppercase()
          .alias("Pack"),

        # Add the dynamic Shop identifier
        pl.lit(shop_name).alias("Retailer"),
    ])

# Apply the transformation
df_trans_7_eleven = parse_product_names_TH(df_prep_7_eleven.unique(), "7-Eleven")

# Display the results
output = df_trans_7_eleven.select([
    "name",
    "promotion_price",
    "original_price",
    "condition",
    "Date",
    "Brand",
    "Volume",
    "Unit",
    "Pack",
    "Retailer"
]).head(15)

print(output)

shape: (12, 10)
┌────────────────┬───────────────┬───────────────┬───────────┬───┬────────┬──────┬──────┬──────────┐
│ name           ┆ promotion_pri ┆ original_pric ┆ condition ┆ … ┆ Volume ┆ Unit ┆ Pack ┆ Retailer │
│ ---            ┆ ce            ┆ e             ┆ ---       ┆   ┆ ---    ┆ ---  ┆ ---  ┆ ---      │
│ str            ┆ ---           ┆ ---           ┆ null      ┆   ┆ f64    ┆ str  ┆ str  ┆ str      │
│                ┆ f64           ┆ f64           ┆           ┆   ┆        ┆      ┆      ┆          │
╞════════════════╪═══════════════╪═══════════════╪═══════════╪═══╪════════╪══════╪══════╪══════════╡
│ ไฮยีน           ┆ 114.0         ┆ 120.0         ┆ null      ┆ … ┆ 480.0  ┆ มล.  ┆ null ┆ 7-Eleven │
│ น้ำยาปรับผ้านุ่ม    ┆               ┆               ┆           ┆   ┆        ┆      ┆      ┆          │
│ มิลค์กี…          ┆               ┆               ┆           ┆   ┆        ┆      ┆      ┆          │
│ เปา ไวท์นาโนเทค ┆ 105.0         ┆ 155.0         ┆ null      ┆ … 

In [12]:
df_trans_7_eleven

name,promotion_price,original_price,condition,Date,Brand,Volume,Unit,Pack,Retailer
str,f64,f64,null,str,str,f64,str,str,str
"""ไฮยีน น้ำยาปรับผ้านุ่ม มิลค์กี…",114.0,120.0,null,"""2026-05-06""","""ไฮยีน""",480.0,"""มล.""",null,"""7-Eleven"""
"""เปา ไวท์นาโนเทค ผงซักฟอก 2400 …",105.0,155.0,null,"""2026-05-06""","""เปา""",2400.0,"""กรัม""",null,"""7-Eleven"""
"""โปร บลูพลัส ผงซักฟอก 2400 กรัม""",99.0,150.0,null,"""2026-05-06""","""โปร""",2400.0,"""กรัม""",null,"""7-Eleven"""
"""ไฮยีน เอ็กซ์เพิร์ท วอช น้ำยาซั…",null,139.0,null,"""2026-05-06""","""ไฮยีน""",1400.0,"""มล.""",null,"""7-Eleven"""
"""ไฮยีน เอ็กซ์เพิร์ทแคร์ น้ำยาปร…",null,129.0,null,"""2026-05-06""","""ไฮยีน""",1000.0,"""มล.""",null,"""7-Eleven"""
…,…,…,…,…,…,…,…,…,…
"""ไฟน์ไลน์ พลัส น้ำยาซักผ้า ซันน…",149.0,179.0,null,"""2026-05-06""","""ไฟน์ไลน์""",1250.0,"""มล.""",null,"""7-Eleven"""
"""แอทแทค ผงซักฟอก แฮปปี้สวีท 250…",null,310.0,null,"""2026-05-06""","""แอทแทค""",2500.0,"""กรัม""",null,"""7-Eleven"""
"""ไฟน์ไลน์พลัสซักผ้าชนิดน้ำสีทอง…",69.0,89.0,null,"""2026-05-06""","""ไฟน์ไลน์""",550.0,"""มล.""",null,"""7-Eleven"""


In [13]:
df_trans_7_eleven.write_excel(f"7_eleven_laundry_dish_{today_date}.xlsx")

## process watchlist

In [29]:
df_prep_7_eleven_watchlist = re_evaluate_price(df_watchlist_sel)
df_trans_7_eleven_watchlist = parse_product_names_TH(df_prep_7_eleven_watchlist.unique(), "7-Eleven")
df_trans_7_eleven_watchlist.write_excel(f"7_eleven_watchlist_{today_date}.xlsx")

In [30]:
df_trans_7_eleven_watchlist.to_pandas()

,name,promotion_price,original_price,condition,Date,Brand,Volume,Unit,Pack,Retailer
0,Unknown Name,NaN,NaN,None,2026-05-06,Unknown,NaN,None,None,7-Eleven
1,ไฮยีน เอ็กซ์เพิร์ทแคร์ น้ำยาปรับผ้านุ่ม ขาวมิล...,NaN,129.0,None,2026-05-06,ไฮยีน,1000.0,มล.,None,7-Eleven
2,ไฮยีน น้ำยาปรับผ้านุ่ม มิลค์กี้ทัช 480 มล.,114.0,120.0,None,2026-05-06,ไฮยีน,480.0,มล.,None,7-Eleven
3,ไฟน์ไลน์ พลัส น้ำยาซักผ้า ซันนี่ โกลด์ 1250 มล.,149.0,179.0,None,2026-05-06,ไฟน์ไลน์,1250.0,มล.,None,7-Eleven
4,แอทแทค ผงซักฟอก แฮปปี้สวีท 2500 กรัม,NaN,310.0,None,2026-05-06,แอทแทค,2500.0,กรัม,None,7-Eleven
5,เปาวินวอช น้ำยาซักผ้าลิควิด 620 มล.,189.0,198.0,None,2026-05-06,เปา,620.0,มล.,None,7-Eleven
6,ไฟน์ไลน์พลัสซักผ้าชนิดน้ำสีทอง 550 มล.,69.0,89.0,None,2026-05-06,ไฟน์ไลน์,550.0,มล.,None,7-Eleven
7,เปา วินวอชลิควิด ถุงเติม น้ำยาซักผ้า 1500 มล.,NaN,185.0,None,2026-05-06,เปา,1500.0,มล.,None,7-Eleven
8,เปา ไวท์นาโนเทค ผงซักฟอก 2400 กรัม,105.0,155.0,None,2026-05-06,เปา,2400.0,กรัม,None,7-Eleven
9,ไลปอนเอฟ น้ำยาล้างจาน สูตรอนามัย 3200 มล.,NaN,165.0,None,2026-05-06,ไลปอนเอฟ,3200.0,มล.,None,7-Eleven
